# Preamble

## Package imports

In [61]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

## Configs

In [62]:
chip_folder = Path("OVChipkaart_Data")

## Function imports

### iter_row_pairs(df)

In [63]:
def iter_row_pairs(df):
    rows = list(df.itertuples(index=True, name="Row"))
    for i in range(0, len(rows) - 1, 2):
        yield rows[i], rows[i + 1]
    if len(rows) % 2 == 1:
        yield rows[-1], None

### checkDuplicate(df)

In [64]:
def checkDuplicate(df: pd.DataFrame) -> bool:
    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError("checkDuplicate(df) expects a pandas DataFrame")

    dups_mask = df.duplicated(subset= ["Datum", "Vertrektijd"], keep=False)

    if not bool(dups_mask.any()):
        return False
    else:
        return True

In [65]:
def showDuplicate(df: pd.DataFrame):
    dups_mask = df.duplicated(subset= ["Datum", "Vertrektijd"], keep=False)
    dup_rows = df.loc[dups_mask]
    dup_counts = (
        dup_rows.groupby(list(df.columns), dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    return dup_rows, dup_counts

In [66]:
showDuplicate(problematic_chip_data)[0]

## CSV import

In [67]:
csv_files = sorted([p for p in chip_folder.glob("*.csv") if p.is_file()])

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in folder: {chip_folder.resolve()}")

In [68]:
chip_data = pd.DataFrame()
for file in csv_files:
    data = pd.read_csv(file, sep=";")
    chip_data = pd.concat([data, chip_data], ignore_index=True)

clean_chip_data = chip_data.drop_duplicates().drop(["Naam", "Kaartnummer", "Product", "Opmerkingen", "Bedrag", "Klasse"], axis=1)

# Preprocessing


In [69]:
merged_chip_data = pd.DataFrame(columns=["Datum", "Vertrektijd", "Vertrekstation", "Aankomsttijd", "Aankomststation"])
problematic_chip_data = pd.DataFrame(columns=["Datum", "Vertrektijd", "Vertrekstation", "Aankomsttijd", "Aankomststation", "TransactionType"])

for r1, r2 in iter_row_pairs(clean_chip_data):
    row_to_add = [r1[1], r1[2], r1[3], r2[4], r2[5]]
    if r2 is None:
        print("Adding to problematic chip data")
        problematic_chip_data.loc[len(problematic_chip_data)] = [r1[1], r1[2], r1[3], r1[4], r1[5], r1[6]]
        raise AssertionError("Odd number of rows, please check input data")
        continue
    if r1[3] != r2[3]:
        #raise AssertionError(f"Departure stations do not match {r1[3]} -- {r2[3]}")
        print("Adding to problematic chip data")
        problematic_chip_data.loc[len(problematic_chip_data)] = [r1[1], r1[2], r1[3], r1[4], r1[5], r1[6]]
        problematic_chip_data.loc[len(problematic_chip_data)] = [r2[1], r2[2], r2[3], r2[4], r2[5], r2[6]]
    if r1[6] == r2[6]:
        #raise AssertionError(f"Transaction types match {r1[6]} -- {r2[6]}")
        print("Adding to problematic chip data")
        problematic_chip_data.loc[len(problematic_chip_data)] = [r1[1], r1[2], r1[3], r1[4], r1[5], r1[6]]
        problematic_chip_data.loc[len(problematic_chip_data)] = [r2[1], r2[2], r2[3], r2[4], r2[5], r2[6]]
    if pd.isna(row_to_add).any():
        print("Adding to problematic chip data")
        problematic_chip_data.loc[len(problematic_chip_data)] = [r1[1], r1[2], r1[3], r1[4], r1[5], r1[6]]
        problematic_chip_data.loc[len(problematic_chip_data)] = [r2[1], r2[2], r2[3], r2[4], r2[5], r2[6]]
    else:
        assert not checkDuplicate(merged_chip_data)
        merged_chip_data.loc[len(merged_chip_data)] = row_to_add
        print("Adding to good chip data")

In [59]:
merged_chip_data

# Visualisation

In [11]:
top_n = 10

dep_counts = merged_chip_data["Vertrekstation"].astype("string").fillna("Onbekend").value_counts()
arr_counts = merged_chip_data["Aankomststation"].astype("string").fillna("Onbekend").value_counts()

dep_top = dep_counts.head(top_n)
dep_other = dep_counts.iloc[top_n:].sum()
if dep_other:
    dep_top.loc["Overig"] = dep_other

arr_top = arr_counts.head(top_n)
arr_other = arr_counts.iloc[top_n:].sum()
if arr_other:
    arr_top.loc["Overig"] = arr_other

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].pie(dep_top.values, labels=dep_top.index, autopct="%1.1f%%", startangle=90)
axes[0].set_title(f"Vertrekstation (top {top_n} + Overig)")
axes[0].axis("equal")

axes[1].pie(arr_top.values, labels=arr_top.index, autopct="%1.1f%%", startangle=90)
axes[1].set_title(f"Aankomststation (top {top_n} + Overig)")
axes[1].axis("equal")

plt.tight_layout()
plt.show()